# Laboratorio · Particiones, cache y persist en Spark (SOLUCIONARIO)

**Capítulo 9 · Spark SQL avanzado y optimización**

En este laboratorio no vas a copiar y pegar: **primero predices, luego ejecutas y luego verificas**. Cada ejercicio tiene una variable con `None` donde escribes tu respuesta *antes* de correr la celda que mide el resultado real.

Lo que vas a practicar:

1. Calcular a mano cuántas particiones crea Spark al leer un archivo de 1 millón de filas.
2. Ver qué pasa con las particiones después de un shuffle y cómo elegir `spark.sql.shuffle.partitions`.
3. Diferenciar `repartition` y `coalesce`, y detectar skew.
4. Decidir cuándo usar `cache`, `persist` y `unpersist`, y verlo en la pestaña **Storage** de Spark UI.
5. Dimensionar particiones para un clúster real de varios nodos.

**Regla del laboratorio:** ten Spark UI abierta todo el tiempo (la URL aparece al ejecutar la primera celda). Cada ejercicio etiqueta sus jobs con `setJobDescription`, así los encuentras fácil en la pestaña **Jobs**.

## 0. Preparación

In [ ]:
import time, math, glob, os
from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel

NUCLEOS = 4          # ajústalo a los núcleos de tu VM (comando: nproc)
RUTA = "/tmp/lab_spark"

spark = (SparkSession.builder
         .master(f"local[{NUCLEOS}]")
         .appName("lab-particiones-cache")
         .config("spark.driver.memory", "2g")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.adaptive.enabled", "false")   # lo activamos en la parte 2
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Spark", spark.version)
print("Núcleos (defaultParallelism):", sc.defaultParallelism)
print("Spark UI:", sc.uiWebUrl)

Ejecuta esta celda sin modificarla. Define las funciones de apoyo que usaremos todo el laboratorio.

In [ ]:
def n_part(df):
    """Número de particiones de un DataFrame."""
    return df.rdd.getNumPartitions()

def filas_por_particion(df):
    """Lista con la cantidad de filas en cada partición (incluye las vacías)."""
    conteo = dict(df.select(F.spark_partition_id().alias("p")).groupBy("p").count().collect())
    return [conteo.get(i, 0) for i in range(n_part(df))]

def tiene_shuffle(df):
    """True si el plan físico contiene un Exchange (shuffle)."""
    return "Exchange" in df._jdf.queryExecution().executedPlan().toString()

def tamano_mb(ruta):
    archivos = [a for a in glob.glob(f"{ruta}/part-*")]
    return sum(os.path.getsize(a) for a in archivos) / 1024**2, len(archivos)

def estado_cache():
    """Muestra lo que hay en cache (lo mismo que la pestaña Storage de Spark UI)."""
    infos = list(sc._jsc.sc().getRDDStorageInfo())
    if not infos:
        print("No hay nada materializado en cache.")
        return (0, 0)
    for i in infos:
        print(f"Particiones en cache: {i.numCachedPartitions()}/{i.numPartitions()} | "
              f"memoria: {i.memSize()/1024**2:.1f} MB | disco: {i.diskSize()/1024**2:.1f} MB")
    return (infos[0].numCachedPartitions(), infos[0].numPartitions())

def cronometro(etiqueta, fn):
    sc.setJobDescription(etiqueta)
    t0 = time.time(); fn(); dt = time.time() - t0
    print(f"{etiqueta}: {dt:.2f} s")
    return dt

def verificar(ej, tu_respuesta, real, explicacion="", tolerancia=0):
    if tu_respuesta is None:
        print(f"[{ej}] Primero escribe tu predicción en la variable y vuelve a ejecutar.")
        return
    if isinstance(real, (int, float)) and not isinstance(real, bool) and tolerancia:
        ok = abs(tu_respuesta - real) <= tolerancia
    else:
        ok = (str(tu_respuesta).strip().lower() == str(real).strip().lower())
    print(f"[{ej}] Tu respuesta: {tu_respuesta} | Real: {real} -> {'✅ Correcto' if ok else '❌ No coincide'}")
    if explicacion:
        print("   ", explicacion)

### Generar 1 millón de ventas

Creamos el dataset y lo guardamos dos veces: como **un solo CSV** y como **Parquet**. El 30% de las ventas pertenece al cliente `0` ("consumidor final"): lo usaremos para ver skew.

In [ ]:
paises = F.array(*[F.lit(p) for p in ["CO", "PE", "EC", "CL", "BR"]])
ventas = (spark.range(1_000_000).withColumnRenamed("id", "id_venta")
    .withColumn("id_cliente", F.when(F.rand(1) < 0.3, F.lit(0))
                               .otherwise((F.rand(2) * 50_000).cast("int")))
    .withColumn("id_producto", (F.rand(3) * 500).cast("int"))
    .withColumn("fecha", F.expr("date_add('2025-01-01', cast(rand(4) * 545 as int))"))
    .withColumn("monto", F.round(F.rand(5) * 2000, 2))
    .withColumn("pais", F.element_at(paises, (F.rand(6) * 5).cast("int") + 1)))

sc.setJobDescription("0 · Generar datos")
ventas.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{RUTA}/ventas_csv")
ventas.write.mode("overwrite").parquet(f"{RUTA}/ventas_parquet")

clientes = (spark.range(50_001).withColumnRenamed("id", "id_cliente")
    .withColumn("segmento", F.element_at(F.array(F.lit("A"), F.lit("B"), F.lit("C")),
                                         (F.rand(7) * 3).cast("int") + 1)))
clientes.write.mode("overwrite").parquet(f"{RUTA}/clientes")

for nombre in ["ventas_csv", "ventas_parquet", "clientes"]:
    mb, n = tamano_mb(f"{RUTA}/{nombre}")
    print(f"{nombre:15} {mb:7.1f} MB en {n} archivo(s)")

## 1. ¿Cuántas particiones crea Spark al leer?

### Ejercicio 1.1 · Datos generados en memoria

`spark.range(1_000_000)` no lee ningún archivo. ¿Cuántas particiones crees que tiene?

*Pista: mira el valor de `defaultParallelism` que imprimió la primera celda.*

In [ ]:
pred_1_1 = sc.defaultParallelism

verificar("1.1", pred_1_1, n_part(spark.range(1_000_000)),
          "Sin archivos de por medio, Spark usa spark.default.parallelism = núcleos disponibles.")

### Ejercicio 1.2 · Calcula a mano las particiones del CSV

Cuando Spark lee archivos, el tamaño de cada partición **no siempre es 128 MB**. La fórmula real es:

```
bytes_por_nucleo = (tamaño_total + n_archivos × openCostInBytes) / núcleos
tamaño_split     = min(maxPartitionBytes, max(openCostInBytes, bytes_por_nucleo))
particiones      = ⌈ tamaño_archivo / tamaño_split ⌉        (para un solo archivo)
```

Valores por defecto: `maxPartitionBytes` = 128 MB y `openCostInBytes` = 4 MB.

Usa el tamaño del CSV que imprimió la celda de generación. **Haz la cuenta en papel o en la celda**, no leas el archivo todavía.

In [ ]:
tam_csv_mb, archivos_csv = tamano_mb(f"{RUTA}/ventas_csv")
print(f"CSV: {tam_csv_mb:.2f} MB en {archivos_csv} archivo | núcleos: {sc.defaultParallelism}")

bytes_por_nucleo = (tam_csv_mb + archivos_csv * 4) / sc.defaultParallelism
tamano_split     = min(128, max(4, bytes_por_nucleo))
pred_1_2         = math.ceil(tam_csv_mb / tamano_split)
print(f"bytes_por_nucleo={bytes_por_nucleo:.2f} MB, split={tamano_split:.2f} MB")

In [ ]:
df_csv = spark.read.option("header", True).csv(f"{RUTA}/ventas_csv")
verificar("1.2", pred_1_2, n_part(df_csv),
          "El archivo pesa menos de 128 MB y aun así se parte: Spark reparte entre los núcleos "
          "para no dejar ninguno ocioso.")

### Ejercicio 1.3 · Cambia `maxPartitionBytes`

Ahora bajamos `maxPartitionBytes` a **4 MB**. Recalcula con la misma fórmula: ¿cuántas particiones habrá?

In [ ]:
pred_1_3 = math.ceil(tam_csv_mb / min(4, max(4, bytes_por_nucleo)))

spark.conf.set("spark.sql.files.maxPartitionBytes", "4m")
df_csv_4m = spark.read.option("header", True).csv(f"{RUTA}/ventas_csv")
parts_1_3 = n_part(df_csv_4m)
verificar("1.3", pred_1_3, parts_1_3,
          "Ahora manda el mínimo: split = min(4 MB, max(4 MB, bytes_por_nucleo)) = 4 MB.")
spark.conf.unset("spark.sql.files.maxPartitionBytes")

### Ejercicio 1.4 · Compruébalo en Spark UI

Ejecuta la celda, abre Spark UI › **Jobs**, busca el job "1.4 · count CSV 4 MB", entra a su stage y cuenta las **tareas** (columna *Tasks*). Escribe ese número.

In [ ]:
spark.conf.set("spark.sql.files.maxPartitionBytes", "4m")
sc.setJobDescription("1.4 · count CSV 4 MB")
spark.read.option("header", True).csv(f"{RUTA}/ventas_csv").count()
spark.conf.unset("spark.sql.files.maxPartitionBytes")

tareas_en_ui = parts_1_3
verificar("1.4", tareas_en_ui, parts_1_3, "Una partición = una tarea en el stage de lectura.")

### Pregunta para pensar 1.5

El Parquet tiene varios archivos pequeños y pesa mucho menos que el CSV porque está comprimido. Antes de ejecutar la siguiente celda, responde en una línea: **¿tendrá más, menos o las mismas particiones que el CSV con la configuración por defecto, y por qué?**

*Tu respuesta:* …

In [ ]:
df_parquet = spark.read.parquet(f"{RUTA}/ventas_parquet")
mb, n = tamano_mb(f"{RUTA}/ventas_parquet")
print(f"Parquet: {mb:.1f} MB en {n} archivos -> {n_part(df_parquet)} particiones")
print(f"CSV:     {tam_csv_mb:.1f} MB en 1 archivo  -> {n_part(df_csv)} particiones")

## 2. Particiones después de un shuffle

### Ejercicio 2.1 · El famoso 200

Con AQE **desactivado**, agrupamos por país. ¿Cuántas particiones tendrá el resultado?

In [ ]:
pred_2_1 = 200

por_pais = df_parquet.groupBy("pais").agg(F.sum("monto").alias("total"))
verificar("2.1", pred_2_1, n_part(por_pais),
          "Todo shuffle produce spark.sql.shuffle.partitions particiones (200 por defecto).")

### Ejercicio 2.2 · ¿Cuántas de esas 200 tienen datos?

Hay 5 países. ¿Cuántas particiones **no vacías** esperas? Después mira en Spark UI el job "2.2": ¿cuántas tareas lanzó el segundo stage para tan pocos datos?

In [ ]:
pred_2_2 = 5

sc.setJobDescription("2.2 · groupBy país con 200 particiones")
filas = filas_por_particion(por_pais)
no_vacias = sum(1 for f in filas if f > 0)
verificar("2.2", pred_2_2, no_vacias,
          f"{len(filas) - no_vacias} tareas se lanzaron para no procesar nada. Eso es overhead puro.")

### Ejercicio 2.3 · Elige tú el número de particiones del shuffle

Regla práctica:

```
particiones_shuffle = max(núcleos, ⌈ MB escritos en el shuffle / 128 ⌉)
```

El tamaño del shuffle lo ves en Spark UI › **Stages** › columna *Shuffle Write*. Ejecuta la agregación por cliente, revisa ese valor en el job "2.3 · medir shuffle" y calcula.

In [ ]:
por_cliente = df_parquet.groupBy("id_cliente").agg(F.sum("monto").alias("total"))
sc.setJobDescription("2.3 · medir shuffle")
por_cliente.write.mode("overwrite").format("noop").save()

shuffle_write_mb = 2       # aprox.: cualquier valor de pocos MB lleva al mismo resultado
pred_2_3 = max(sc.defaultParallelism, math.ceil(shuffle_write_mb / 128))

In [ ]:
recomendado = max(sc.defaultParallelism, math.ceil((shuffle_write_mb or 0) / 128))
verificar("2.3", pred_2_3, recomendado,
          "Con datos pequeños el mínimo razonable son los núcleos: una tarea por núcleo, una sola ola.")

spark.conf.set("spark.sql.shuffle.partitions", recomendado)
sc.setJobDescription("2.3 · con particiones ajustadas")
por_cliente = df_parquet.groupBy("id_cliente").agg(F.sum("monto").alias("total"))
print("Particiones ahora:", n_part(por_cliente))

### Ejercicio 2.4 · Deja que AQE decida

Volvemos a 200 particiones y activamos AQE. AQE mide el shuffle real y **une particiones pequeñas**. ¿Con cuántas particiones crees que termina la agregación por país?

*Responde con un número; si no sabes, piensa: ¿necesita AQE más de una partición para 5 filas?*

In [ ]:
pred_2_4 = 1

spark.conf.set("spark.sql.shuffle.partitions", 200)
spark.conf.set("spark.sql.adaptive.enabled", "true")
sc.setJobDescription("2.4 · groupBy país con AQE")
por_pais_aqe = df_parquet.groupBy("pais").agg(F.sum("monto").alias("total"))
verificar("2.4", pred_2_4, n_part(por_pais_aqe),
          "AQE coalesce: une particiones post-shuffle hasta un tamaño objetivo (64 MB por defecto).")

In [ ]:
# Volvemos a una configuración controlada para el resto del laboratorio
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", 8)

## 3. repartition, coalesce y skew

### Ejercicio 3.1 · ¿Quién hace shuffle?

Responde `True` o `False` antes de ejecutar.

In [ ]:
pred_3_1_coalesce    = False
pred_3_1_repartition = True

verificar("3.1a", pred_3_1_coalesce, tiene_shuffle(df_parquet.coalesce(2)),
          "coalesce fusiona particiones vecinas sin mover datos por red.")
verificar("3.1b", pred_3_1_repartition, tiene_shuffle(df_parquet.repartition(2)),
          "repartition siempre redistribuye todo: Exchange en el plan.")

### Ejercicio 3.2 · La trampa de coalesce

`df_parquet` tiene las particiones que viste en la pregunta 1.5. ¿Cuántas particiones tendrá `df_parquet.coalesce(50)`?

In [ ]:
pred_3_2 = n_part(df_parquet)

verificar("3.2", pred_3_2, n_part(df_parquet.coalesce(50)),
          "coalesce solo puede reducir. Para subir particiones necesitas repartition.")

### Ejercicio 3.3 · Detecta el skew

Repartimos por `id_cliente` en 8 particiones. Recuerda que el 30% de las ventas es del cliente `0` y el 70% restante se reparte entre 50.000 clientes.

¿Qué **porcentaje** de las filas tendrá la partición más grande? (número entre 0 y 100; se acepta ±4)

In [ ]:
pred_3_3 = 30 + 70 / 8

sc.setJobDescription("3.3 · repartition por id_cliente")
filas = filas_por_particion(df_parquet.repartition(8, "id_cliente"))
pct_max = round(100 * max(filas) / sum(filas), 1)
print("Filas por partición:", filas)
verificar("3.3", pred_3_3, pct_max,
          "≈ 30% del cliente 0 + 70%/8 de los demás. Una tarea hace ~4 veces el trabajo de las otras.",
          tolerancia=4)

Ahora abre Spark UI › **Stages**, entra al stage del job "3.3" y mira *Summary Metrics*: compara **Max** con **Median** en *Duration* y *Shuffle Read Size*. Así se ve el skew en producción.

Compara con `repartition(8)` sin columna (round robin):

In [ ]:
print("Sin columna:", filas_por_particion(df_parquet.repartition(8)))

### Ejercicio 3.4 · Controla los archivos de salida

Completa el código para que la salida quede en **exactamente 3 archivos**, sin shuffle.

In [ ]:
salida = f"{RUTA}/salida_3_4"
df_parquet.coalesce(3).write.mode("overwrite").parquet(salida)

_, n_archivos = tamano_mb(salida) if os.path.exists(salida) else (0, 0)
verificar("3.4", n_archivos if n_archivos else None, 3, "coalesce(3): una partición = un archivo.")

## 4. cache, persist y unpersist

Construimos un DataFrame **caro**: lee ventas, filtra, une con clientes y agrega. Luego lo usamos en tres acciones.

In [ ]:
base = (spark.read.parquet(f"{RUTA}/ventas_parquet")
        .filter("fecha >= '2025-06-01'")
        .join(spark.read.parquet(f"{RUTA}/clientes"), "id_cliente")
        .groupBy("id_cliente", "segmento", "pais")
        .agg(F.sum("monto").alias("total"), F.count("*").alias("compras")))

def tres_acciones(df, etiqueta):
    t = 0
    t += cronometro(f"{etiqueta} · acción 1 count", lambda: df.count())
    t += cronometro(f"{etiqueta} · acción 2 por segmento", lambda: df.groupBy("segmento").sum("total").collect())
    t += cronometro(f"{etiqueta} · acción 3 top", lambda: df.orderBy(F.desc("total")).limit(5).collect())
    return t

t_sin_cache = tres_acciones(base, "4.1 sin cache")
print(f"Total sin cache: {t_sin_cache:.2f} s")

### Ejercicio 4.1 · Cuenta en Spark UI

En la pestaña **Jobs**, mira los jobs "4.1 sin cache". ¿Cuántas veces aparece la lectura del parquet de ventas (FileScan) sumando las tres acciones? Abre el DAG de cada job para comprobarlo.

In [ ]:
pred_4_1 = 3
verificar("4.1", pred_4_1, 3, "Sin cache, cada acción recalcula todo el linaje: lectura, filtro, join y agregación.")

### Ejercicio 4.2 · ¿Ya está en cache?

Ejecutamos `base.cache()` **sin ninguna acción**. Predice las dos cosas:

In [ ]:
pred_4_2_flag       = True
pred_4_2_particiones = 0

base.cache()
cacheadas, total = estado_cache()
verificar("4.2a", pred_4_2_flag, base.is_cached, "is_cached solo dice que el DataFrame está MARCADO.")
verificar("4.2b", pred_4_2_particiones, cacheadas,
          "cache es lazy: hasta que no haya una acción, no se guarda nada. Mira la pestaña Storage: vacía.")

### Ejercicio 4.3 · Cache parcial

Ahora ejecutamos `base.take(1)`. Ese DataFrame tiene 8 particiones (por `shuffle.partitions = 8`). ¿Cuántas quedan en cache?

In [ ]:
pred_4_3 = 1

sc.setJobDescription("4.3 · take(1)")
base.take(1)
cacheadas, total = estado_cache()
verificar("4.3", pred_4_3, cacheadas,
          "take(1) solo calcula las particiones que necesita para devolver una fila. "
          "En Spark UI › Storage verás 'Fraction Cached' menor a 100%.")

### Ejercicio 4.4 · Materializa y mide

Completa la línea con la acción que materializa **todas** las particiones. Luego compara tiempos.

In [ ]:
sc.setJobDescription("4.4 · materializar cache")
base.count()

cacheadas, total = estado_cache()
print("¿Cache completo?", cacheadas == total)

In [ ]:
t_con_cache = tres_acciones(base, "4.4 con cache")
print(f"Sin cache: {t_sin_cache:.2f} s | Con cache: {t_con_cache:.2f} s | "
      f"mejora: {t_sin_cache / max(t_con_cache, 0.01):.1f}x")

Abre el DAG de cualquier job "4.4 con cache": en lugar del FileScan ahora verás **InMemoryTableScan**. También puedes confirmarlo con `base.explain()`.

### Ejercicio 4.5 · Cambia el nivel de almacenamiento

Libera la memoria y vuelve a persistir `base` **solo en disco**. Completa las dos líneas.

In [ ]:
base.unpersist()
base.persist(StorageLevel.DISK_ONLY)

pred_4_5 = "disco"

sc.setJobDescription("4.5 · persist DISK_ONLY")
base.count()
estado_cache()
print("Nivel:", base.storageLevel)

In [ ]:
nivel_real = "disco" if base.storageLevel.useDisk and not base.storageLevel.useMemory else "memoria"
verificar("4.5", pred_4_5, nivel_real,
          "DISK_ONLY es útil en VMs con poca RAM: no compite con la memoria de joins y agregaciones.")

### Ejercicio 4.6 · ¿Dónde va el persist?

No ejecutes nada: lee cada escenario y responde. En los escenarios, `acción` significa `count`, `write`, `show` o `collect`.

**Escenario A**
```python
raw    = spark.read.parquet("ventas")                  # A
limpio = raw.filter("monto > 0")                       # B
enriq  = limpio.join(clientes, "id_cliente")           # C
enriq.groupBy("pais").sum("monto").write.parquet(...)      # acción 1
enriq.groupBy("segmento").sum("monto").write.parquet(...)  # acción 2
```
¿Qué DataFrame persistes? Responde `A`, `B`, `C` o `ninguno`.

**Escenario B**
```python
reporte = spark.read.parquet("ventas").groupBy("pais").sum("monto")
reporte.write.parquet("salida")
```
¿Qué persistes? (`reporte` o `ninguno`)

**Escenario C** · Un resultado de join de 50 GB que se usa 3 veces, en un clúster con 20 GB de memoria total para cache. ¿Qué nivel usas? (`MEMORY_ONLY`, `MEMORY_AND_DISK` o `ninguno`)

**Escenario D**
```python
features = preparar(df).persist()       # línea 1
for i in range(20):                     # línea 2
    modelo = entrenar(features, i)      # línea 3
guardar(modelo)                         # línea 4
```
¿Después de qué línea va `features.unpersist()`? (número de línea)

In [ ]:
resp_A = "C"
resp_B = "ninguno"
resp_C = "MEMORY_AND_DISK"
resp_D = 3

In [ ]:
verificar("4.6 A", resp_A, "C", "C es donde el linaje se ramifica: dos acciones lo usan.")
verificar("4.6 B", resp_B, "ninguno", "Una sola acción: cachear solo gasta memoria y tiempo.")
verificar("4.6 C", resp_C, "MEMORY_AND_DISK",
          "MEMORY_ONLY recalcularía lo que no cabe; con MEMORY_AND_DISK lo que no cabe va a disco local.")
verificar("4.6 D", resp_D, 3,
          "Se libera al terminar el bucle (antes de la línea 4): ya nadie más lo usa.")

In [ ]:
base.unpersist()
spark.catalog.clearCache()
estado_cache()

## 5. Dimensionar particiones en un clúster

En tu VM todo corre en un solo nodo. En producción hay varios nodos, cada uno con varios executors, y cada executor tiene varios núcleos. **Cada núcleo ejecuta una tarea a la vez**: eso es un *slot*.

```
slots = nodos × executors_por_nodo × núcleos_por_executor
olas  = ⌈ particiones / slots ⌉
```

Una **ola** es una ronda de tareas en paralelo. Si la última ola queda casi vacía, pagas el tiempo de una ola completa con la mayoría de núcleos ociosos.

**Clúster del ejercicio:** 5 nodos workers de 16 núcleos y 64 GB. Se configuran 3 executors por nodo, cada uno con 5 núcleos (el núcleo restante queda para el sistema operativo).

In [ ]:
pred_5_1 = 5 * 3 * 5
pred_5_2 = math.ceil(30 * 1024 / 128)
pred_5_3 = math.ceil(pred_5_2 / pred_5_1)
pred_5_4 = pred_5_3 * pred_5_1 - pred_5_2

In [ ]:
slots = 5 * 3 * 5
parts = math.ceil(30 * 1024 / 128)
olas  = math.ceil(parts / slots)
verificar("5.1", pred_5_1, slots)
verificar("5.2", pred_5_2, parts, "Con muchos datos, bytes_por_nucleo supera 128 MB y manda maxPartitionBytes.")
verificar("5.3", pred_5_3, olas)
verificar("5.4", pred_5_4, olas * slots - parts,
          "La última ola solo tiene 15 tareas: 60 núcleos esperando. Ajustar a un múltiplo de los slots lo evita.")

### Ejercicio 5.5 · Elige el número de particiones

Para esos 30 GB, propone un número de particiones que cumpla dos condiciones:

1. Sea **múltiplo de los slots** (olas completas).
2. Deje cada partición entre **100 y 200 MB**.

In [ ]:
pred_5_5 = 225

In [ ]:
if pred_5_5:
    mb_part = 30 * 1024 / pred_5_5
    ok = pred_5_5 % slots == 0 and 100 <= mb_part <= 200
    print(f"{pred_5_5} particiones -> {mb_part:.0f} MB cada una, {pred_5_5 / slots:.1f} olas -> "
          f"{'✅ Cumple' if ok else '❌ Revisa: múltiplo de ' + str(slots) + ' y entre 100 y 200 MB'}")
    print("Se aplica con df.repartition(n) antes de la operación pesada, o con spark.sql.shuffle.partitions.")
else:
    print("Primero escribe tu propuesta en pred_5_5.")

### Ejercicio 5.6 · Tu propia VM

Con lo aprendido, ¿cuántas particiones de shuffle recomiendas para este laboratorio? Usa la regla `max(núcleos, ⌈ MB / 128 ⌉)` con el tamaño del Parquet de ventas.

In [ ]:
pred_5_6 = max(sc.defaultParallelism, math.ceil(tamano_mb(f"{RUTA}/ventas_parquet")[0] / 128))

In [ ]:
mb_parquet = tamano_mb(f"{RUTA}/ventas_parquet")[0]
verificar("5.6", pred_5_6, max(sc.defaultParallelism, math.ceil(mb_parquet / 128)),
          "Con 1M de filas, los datos son pequeños: lo que limita es el número de núcleos, no el tamaño.")

## Cierre

Antes de terminar, responde en tu cuaderno:

1. ¿Por qué un CSV de menos de 128 MB se leyó en varias particiones?
2. ¿Qué viste en la pestaña **Storage** después de `take(1)` y después de `count()`?
3. ¿En qué punto de un pipeline pones un `persist`, y cuándo haces `unpersist`?
4. Si tu clúster tiene 120 slots y 1 TB de datos, ¿cuántas particiones propones?

Para seguir experimentando con clústeres de distintos tamaños, usa el simulador de particiones que compartió el docente.

In [ ]:
spark.stop()